In [0]:
from pyspark.sql import functions as F

In [0]:
# Read the table from samples database

cus_df = spark.table("samples.bakehouse.sales_customers")
cus_df.display()

In [0]:
cus2_df = spark.sql("select customerID, first_name, last_name from samples.bakehouse.sales_customers limit 10")
cus2_df.display()

In [0]:
cus_df.createOrReplaceTempView("customers")

In [0]:
%sql
-- Find out customer count from each country
select country, count(customerID) as cus_count from customers group by country;

In [0]:
res_df = (
    cus_df
    .groupBy("country")
    .agg(F.count("customerID").alias("cus_count"))
)

display(res_df)

In [0]:
cus_df = (
    cus_df
    .withColumn(
        "gender_identifier",
        F.when(F.col("gender") == "female", 0)
        .otherwise(1)
    )
)
display(cus_df)

In [0]:
# Find out the gender ration by country

ratio_df = (
    cus_df
    .groupBy('country')
    .agg(
        (((F.sum(F.when(F.col("gender") == "male", 1).otherwise(0))) / (F.sum(F.when(F.col("gender") == "female", 1).otherwise(0))))).alias("fm_ratio")
    )
)

display(ratio_df)

In [0]:
ratio_df = (
    cus_df
    .groupBy('country')
    .agg(
        ((F.sum(F.when(F.col("gender") == "male", 1).otherwise(0))) / (F.count("*"))).alias("male_ratio"),
        ((F.sum(F.when(F.col("gender") == "female", 1).otherwise(0))) / (F.count("*"))).alias("female_ratio")

    )
)

display(ratio_df)

In [0]:
ratio_df = (
    cus_df
    .groupBy('country')
    .agg(
        F.count("*").alias("TotalPeople"),
        (F.sum(F.when(F.col("gender") == "male", 1).otherwise(0))).alias("males"),
        (F.sum(F.when(F.col("gender") == "female", 1).otherwise(0))).alias("females"),
        (((F.sum(F.when(F.col("gender") == "male", 1).otherwise(0))) / (F.count("*")))*100).alias("male_ratio"),
        (((F.sum(F.when(F.col("gender") == "female", 1).otherwise(0))) / (F.count("*")))*100).alias("female_ratio")
    )
)
ratio_df.display()

In [0]:
ratio_df = (
    ratio_df
    .withColumns({
        "male_ratio": F.concat(F.round(F.col("male_ratio"), 2), F.lit("%")),
        "female_ratio": F.concat(F.round(F.col("female_ratio"), 2), F.lit("%"))
    })
)
ratio_df.display()

In [0]:
ratio_df.write.mode("overwrite").format("csv").option("header", True).save("/Volumes/quant_databricks/batch0506/quantcloudrawdatasets/customers")

In [0]:
ratio_df.coalesce(1).write.mode("overwrite").format("csv").option("header", True).save("/Volumes/quant_databricks/batch0506/quantcloudrawdatasets/customers")

In [0]:
import pandas as pd

In [0]:
ratio_df.toPandas().to_csv("/Volumes/quant_databricks/batch0506/quantcloudrawdatasets/customers1.csv", index=False)